# Lesson 3 : Thread (Conversation)

By using thread in Agent Framework, you can handle the conversation thread.<br>
In this exercise, we'll learn how to use an agent thread.

## Prepare agent

Same as previous exercises, connect to the existing agent in Microsoft Foundry.

In [1]:
from agent_framework.azure import AzureAIClient
from agent_framework import ChatAgent
from azure.identity.aio import AzureCliCredential
from typing import Annotated
from pydantic import Field
from random import randint

# initialize a client object
credential = AzureCliCredential()
client = AzureAIClient(
    credential=credential,
    agent_name="BasicWeatherAgent",
    agent_version="1",
)

# define local tools
def get_weather(
    location: Annotated[str, Field(description="天気を取得する場所")],  # "the location to get the weather for"
) -> str:
    """与えられた場所の天気を取得する。"""
    conditions = ["晴れ", "曇り", "雨", "嵐"]  # "sunny", "cloudy", "rainy", "stormy"
    return f"{location}の天気は{conditions[randint(0, 3)]}です。"  # "the weather in ... is ... ."

def get_temperature(
    location: Annotated[str, Field(description="気温を取得する場所")],  # "the location to get the temperature for"
) -> str:
    """与えられた場所の気温を取得する。"""
    return f"{location}の気温は{randint(10, 30)}°Cです。"  # "the temperature in ... is ... degrees."

# connect to the agent
agent = ChatAgent(
    chat_client=client,
    instructions="あなたは、気象情報に関する Agent です。",  # "you are an agent about weather information."
    tools=[get_weather, get_temperature])

## Run agent without thread

First, let's run the following example without conversation thread.<br>
In this example, first we ask weather and temperature, and next we ask to create a weather summary.

In Agent Framework, each request statelessly runs by default, and these 2 requests are performed in different agent thread. (The response id is not inherited and tool execution will be called twice internally. Use tracing and see the internal steps.)

In [2]:
from IPython.display import Markdown, display

result = await agent.run("大阪の天気と気温を教えてください。")  # "tell me the weather and temperature in Osaka."
display(Markdown(result.text))

result = await agent.run("大阪の気象に関するサマリーを作成してください。サマリーには、天気概況、注意すべき点の 2 つを記載してください。")  # "write a summary of the weather in Osaka. In the summary, include two things: the general weather conditions and important points to note."
display(Markdown(result.text))

大阪の天気は**晴れ**です。  
大阪の気温は**30°C**です。

## 大阪の気象サマリー

### 1) 天気概況
- 現在の大阪は**嵐**です。気温は**25°C**です。

### 2) 注意すべき点
- **強風・激しい雨**により、外出時の安全確保（飛来物、倒木、視界不良）に注意してください。  
- **交通への影響**（遅延・運休、道路の冠水など）が出る可能性があるため、最新情報の確認をおすすめします。

## Run agent with thread

Now we fix this code.<br>
By assigning same ```AgentThread```  object in ```run()``` method, the agent will now run on the same conversation. (In this example, response id in Azure OpenAI Responses API will be inherited internally. The internal implementation depends on the corresponding client.)

In [3]:
thread = agent.get_new_thread()

result = await agent.run(
    "大阪の天気と気温を教えてください。",  # "tell me the weather and temperature in Osaka."
    thread=thread,
)
display(Markdown(result.text))

result = await agent.run(
    "大阪の気象に関するサマリーを作成してください。サマリーには、天気概況、注意すべき点の 2 つを記載してください。",  # "write a summary of the weather in Osaka. In the summary, include two things: the general weather conditions and important points to note."
    thread=thread,
)
display(Markdown(result.text))

大阪の天気は**晴れ**です。  
大阪の気温は**27°C**です。

## 天気概況
- 大阪は**晴れ**で、気温は**27°C**です。

## 注意すべき点
- **日差しが強い可能性**があるため、外出時は帽子・日焼け止めなどの紫外線対策を。
- **やや暑さを感じやすい気温**なので、こまめな水分補給と適度な休憩を心がけてください。